In [ ]:
import yaml as yml
import pandas as pd
from emu_renewal.constants import BASE_PATH
from emu_renewal.outputs import get_table_df_from_priors_dict
from emu_renewal.plotting import plot_beta_priors, plot_progress_priors
from numpyro import distributions as dist
from IPython.display import Markdown
from matplotlib_inline.backend_inline import set_matplotlib_formats

set_matplotlib_formats("svg")

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from emu_renewal.constants import DUR_MIN, DUR_REL_MAX

In [ ]:
loaded_priors = yml.safe_load(open(BASE_PATH / "data/config/priors.yml", "r"))
duration_params = loaded_priors["durations"]

In [ ]:
# Note that using set() loses the ordering of this list
duration_types = list(dict.fromkeys([p.rsplit("_", 1)[0] for p in duration_params]))

In [ ]:
fig, axes = plt.subplots(len(duration_types), 2, figsize=(15, 18), width_ratios=[2, 1])
for d, dur in enumerate(duration_types):
    mean_param = duration_params[dur + "_mean"]
    sd_param = duration_params[dur + "_sd"]
    
    mean_prior = dist.TruncatedNormal(mean_param["mean"], mean_param["sd"], low=DUR_MIN, high=mean_param["mean"] * DUR_REL_MAX)
    sd_prior = dist.TruncatedNormal(sd_param["mean"], sd_param["sd"], low=DUR_MIN, high=sd_param["mean"] * DUR_REL_MAX)
    
    mean_x_vals = np.linspace(0.0, 30.0, 1000)
    sd_x_vals = np.linspace(0.0, 15.0, 1000)
    mean_y_vals = np.exp(mean_prior.log_prob(mean_x_vals))
    sd_y_vals = np.exp(sd_prior.log_prob(sd_x_vals))
    
    mean_ax = axes[d, 0]
    mean_ax.fill_between(mean_x_vals, mean_y_vals / max(mean_y_vals), color="0.8")
    mean_ax.plot(mean_x_vals, mean_y_vals / max(mean_y_vals), color="k", linewidth=2.0)
    mean_ax.set_title(mean_param["param_name"].replace(" (days)", ""))
    mean_ax.set_xlabel("days")
    mean_ax.set_yticks([])
    
    sd_ax = axes[d, 1]
    sd_ax.fill_between(sd_x_vals, sd_y_vals / max(sd_y_vals), color="0.8")
    sd_ax.plot(sd_x_vals, sd_y_vals / max(sd_y_vals), color="k", linewidth=2.0)
    sd_ax.set_title(sd_param["param_name"].replace(" (days)", ""))
    sd_ax.set_xlabel("days")
    sd_ax.set_yticks([])
    
fig.tight_layout()

In [ ]:
fig.savefig("dur_priors.png")

In [ ]:
col_widths = '{tbl-colwidths="[14, 8, 8, 70]"}'
durations_df = get_table_df_from_priors_dict(loaded_priors["durations"])
keep_cols = [c for c in durations_df if c != "Short_name"]
dur_table = durations_df[keep_cols]
caption = "\n: Parameters and supporting evidence to time period priors. "
Markdown(dur_table.to_markdown() + "\n" + caption + col_widths)

In [ ]:
betas_df = get_table_df_from_priors_dict(loaded_priors["beta"])
caption = "\n: Parameters and supporting evidence to beta-distributed priors. "
Markdown(betas_df.to_markdown() + caption + col_widths)

In [ ]:
#| fig-cap: "Illustration of beta prior distributions"
plot_beta_priors(loaded_priors)